# 02 — IV Characteristic & Breakdown Voltage  


---
## 1. Imports

In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.utils.ingestion import DataIngestionConfig, DataIngestionService
from src.simulator.breakdown import MultiplicationCurrentRise

print('Imports OK')

---
## 2. Build device and find breakdown

In [ ]:
config = DataIngestionConfig.from_defaults()
svc = DataIngestionService(config)
sim = svc.build_simulator()
device = sim.device

print(f"Device: {len(device.layers)} layers, {device.L*1e4:.1f} µm")
print(f"Grid: {device.no_of_nodes} nodes")

---
## 3. The McIntyre multiplication formula

The avalanche multiplication factor $M$ (mean current gain) for a single carrier injected at $x_0$ follows the **McIntyre integral**:

$$
M = \frac{1}{1 - \displaystyle\int_0^W \beta(x)\, \exp\!\left(\int_x^W [\alpha(x') - \beta(x')]\, dx'\right) dx}
$$

### 3.1 Physical meaning

The denominator is the **ionization path integral**. When it approaches 0, $M \to \infty$ — this is **breakdown**. The integral accumulates the probability that a carrier generates at least one e-h pair that goes on to generate more pairs (positive feedback).

### 3.2 Dead-space correction

In thin ($< 1$ µm) multiplication layers, a newly created carrier must travel a minimum distance (the **dead space**) before it gains enough energy from the field to cause ionization. The effective ionization coefficient is:

$$
\alpha_{\text{eff}}(x) = \frac{\alpha(x)}{1 + \alpha(x) \cdot l_d}
$$

where $l_d = E_{\text{th}} / |E(x)|$ is the dead-space length and $E_{\text{th}} \approx 1.5 \times E_g$ is the threshold energy.

### 3.3 Implementation

The `_compute_multiplication` method in the simulator computes $M$ via trapezoidal integration:

In [ ]:
# Manually compute M at several biases to see the blow-up
from src.avalanche.ionization import IonizationCoefficients

biases = np.arange(10, 65, 5)
M_values = []

for V in biases:
    phi, E, info = sim.solve_poisson(float(V))
    E_abs = np.abs(E)
    
    # Use effective coefficients with dead-space correction
    alpha = sim.ionization.effective_alpha_n(E_abs, Eg=1.35)
    beta = sim.ionization.effective_alpha_p(E_abs, Eg=1.35)
    
    x = device.grid.x
    active = E_abs > 1e5  # only high-field region contributes
    
    if not np.any(active):
        M = 1.0
    else:
        dx = np.diff(x)
        diff = alpha - beta
        
        # Backward cumulative sum (from W to 0)
        cum = np.zeros_like(x)
        for i in range(len(x) - 2, -1, -1):
            cum[i] = cum[i + 1] + diff[i] * dx[i]
        
        integrand = beta * np.exp(cum)
        denom = 1.0 - float(np.trapezoid(integrand, x))
        M = 1e6 if denom <= 0.01 else min(1.0 / denom, 1e6)
    
    M_values.append(M)
    print(f"V = {V:5.1f} V  |  denominator = {denom:.4f}  |  M = {M:.1f}")

In [ ]:
# Plot M vs bias
plt.figure(figsize=(8, 4))
plt.semilogy(biases, M_values, 'o-', lw=2)
plt.axvline(60, color='r', ls='--', alpha=0.5, label='Breakdown (Vbr=60 V)')
plt.xlabel('Bias Voltage (V)')
plt.ylabel('Multiplication Factor M')
plt.grid(alpha=0.3)
plt.title('McIntyre multiplication factor vs bias')
plt.legend()
plt.show()

**Key observation:** Below breakdown, M grows slowly. As $V \to V_{BR}$, the denominator approaches 0 and $M$ shoots up. The simulator caps M at $10^6$ to avoid numerical overflow.

---
## 4. Breakdown sweep algorithm

The breakdown detection uses `MultiplicationCurrentRise`:

For each bias $V$:
1. Solve Poisson → $\phi$, $E$
2. Compute $\alpha_{\text{eff}}(E)$, $\beta_{\text{eff}}(E)$ with dead-space
3. Compute $M(V)$ via the McIntyre integral
4. Compute dark current: $I_{\text{total}} = \int J_{\text{dark}}(x)\,dx \times A \times M$
5. If $I_{\text{total}} / I_{\text{prev}} > 5\times$, declare breakdown at $V$

Let's run the full breakdown sweep and see the current history:

In [ ]:
Vbr, history = sim.find_breakdown(V_start=20, V_max=80, V_step=1.0)
print(f"Breakdown voltage: Vbr = {Vbr:.1f} V")
print(f"Number of sweep steps: {len(history)}")

# Extract I_total from history
Vs = [h['V'] for h in history if 'I_total' in h]
Is = [h['I_total'] for h in history if 'I_total' in h]
Ms = [h['M'] for h in history if 'M' in h]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.semilogy(Vs, Is, 'o-', lw=2)
ax1.axvline(Vbr, color='r', ls='--', alpha=0.5, label=f'Vbr = {Vbr} V')
ax1.set_xlabel('Bias (V)')
ax1.set_ylabel('I_total (A)')
ax1.grid(alpha=0.3)
ax1.legend()
ax1.set_title('Total dark current during breakdown sweep')

ax2.semilogy(Vs, Ms, 'o-', lw=2, color='orange')
ax2.axvline(Vbr, color='r', ls='--', alpha=0.5)
ax2.set_xlabel('Bias (V)')
ax2.set_ylabel('M')
ax2.grid(alpha=0.3)
ax2.set_title('Multiplication factor during sweep')

plt.tight_layout()
plt.show()

---
## 5. Dark IV and illuminated IV curves

The full IV characteristic includes:
- **Dark current:** SRH + BTBT + TAT generation, multiplied by M
- **Photocurrent:** Optically generated carriers (1550 nm), multiplied, added to dark current
- **Geiger region:** Above breakdown, a simple $V_{ex}/R_q$ term approximates the self-quenched current

In [ ]:
# Compute IV curve
V_sweep = np.arange(0, Vbr + 25, 1.0)
I_dark_list, I_light_list = [], []
R_Q = 1e5  # 100 k\Omega quench resistor (from _config)

for V in V_sweep:
    try:
        _, E, Pe, Ph, _, xr = sim.get_fields(float(V))
        dc = sim.compute_dark_current(float(V), E=E)
        
        if float(V) >= Vbr:
            V_ex = float(V) - Vbr
            I_geiger = V_ex / R_Q
            I_dark_list.append(dc["I_dark"] + I_geiger)
            I_light_list.append(dc["I_dark"] + I_geiger)  # simplified
        else:
            I_dark_list.append(dc["I_dark"])
            # Photocurrent approximation
            M = min(dc["M"], 2000.0)
            I_photo = 1e-9 * M  # simplified photocurrent estimate
            I_light_list.append(dc["I_dark"] + I_photo)
    except Exception:
        I_dark_list.append(np.nan)
        I_light_list.append(np.nan)

In [ ]:
plt.figure(figsize=(10, 5))
plt.semilogy(V_sweep, np.abs(I_dark_list), 'b-', lw=2, label='Dark current')
plt.semilogy(V_sweep, np.abs(I_light_list), 'orange', lw=2, label='Illuminated (1550 nm)')
plt.axvline(Vbr, color='r', ls='--', alpha=0.5, label=f'Vbr = {Vbr:.1f} V')
plt.xlabel('Reverse Bias (V)')
plt.ylabel('Current (A)')
plt.legend(fontsize=9)
plt.grid(alpha=0.3)
plt.title('I-V Characteristic: Dark and Illuminated')
plt.show()

---
## 6. The 5× current-rise criterion

The `MultiplicationCurrentRise` class tracks $I_{\text{prev}}$ and checks:  
**If $I_{\text{total}} / I_{\text{prev}} > 5$ → breakdown**

This catches the sharp knee in the IV curve. Below breakdown, M grows slowly (10–100× over 10+ volts). At breakdown, M goes from ~100 to >10⁶ in a single 1 V step, giving a >5× current jump.

Let's verify by looking at the actual ratios from our sweep:

In [ ]:
ratios = []
for i in range(1, len(Is)):
    if Is[i-1] > 0:
        ratios.append(Is[i] / Is[i-1])
    else:
        ratios.append(0)

plt.figure(figsize=(8, 4))
plt.plot(Vs[1:], ratios, 'o-', lw=2, color='green')
plt.axhline(5, color='r', ls='--', alpha=0.5, label='5× threshold')
plt.axvline(Vbr, color='orange', ls=':', alpha=0.7, label=f'Vbr = {Vbr} V')
plt.xlabel('Bias (V)')
plt.ylabel('I(V) / I(V−1)')
plt.grid(alpha=0.3)
plt.legend()
plt.title('Current ratio per bias step — the 5× criterion')
plt.yscale('log')
plt.show()

---
## 7. Summary

- The **McIntyre integral** relates the electric field to the multiplication factor $M$.
- **Dead-space correction** matters for thin InP multiplication layers (~0.6 µm).
- Breakdown is detected when the total dark current surges **>5× in one bias step**.
- The IV curve shows a sharp knee at $V_{BR}$ — below it, the SPAD behaves like a linear APD; above it, Geiger-mode current is limited by the quench resistor.

**Next experiment (03):** Impact ionization coefficients — Okuto-Crowell vs Van Overstraeten–de Man, temperature dependence, and the two-regime Chynoweth form.